# 🚗 Car Type Classifier — Notebook 1 : Exploration des données

**Objectif** : Explorer et visualiser les données avant la classification.

**Datasets** :
- [Cars Datasets 2025](https://www.kaggle.com/datasets/abdulmalik1518/cars-datasets-2025)
- [Large Dataset of Cars](https://www.kaggle.com/datasets/makslypko/large-cars-dataset)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
print('✅ Librairies chargées')

## 1. Chargement des données

In [ ]:
# Si vous avez téléchargé les datasets Kaggle, modifiez ce chemin
DATA_DIR = '../data/'

csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')] if os.path.exists(DATA_DIR) else []

if csv_files:
    dfs = [pd.read_csv(os.path.join(DATA_DIR, f), on_bad_lines='skip') for f in csv_files]
    df = pd.concat(dfs, ignore_index=True)
    print(f'✅ {len(df)} lignes chargées depuis {len(csv_files)} fichier(s)')
else:
    print('⚠️ Aucun CSV trouvé — utilisation du dataset de démonstration')
    import sys; sys.path.append('../src')
    from classifier import generate_demo_dataset
    df = generate_demo_dataset()

df.head()

## 2. Informations générales

In [ ]:
print(f'Dimensions : {df.shape}')
print(f'\nColonnes :')
print(df.dtypes)
print(f'\nValeurs manquantes :')
print(df.isnull().sum())

In [ ]:
df.describe()

## 3. Distribution des colonnes clés

In [ ]:
# Distribution du nombre de places
seat_col = next((c for c in df.columns if 'seat' in c.lower() or 'passenger' in c.lower()), None)

if seat_col:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    df[seat_col].dropna().value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C72B0', edgecolor='white')
    axes[0].set_title('Distribution du nombre de places', fontweight='bold')
    axes[0].set_xlabel('Nombre de places')
    axes[0].set_ylabel('Nombre de véhicules')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Distribution puissance
    hp_col = next((c for c in df.columns if 'hp' in c.lower() or 'horse' in c.lower() or 'power' in c.lower()), None)
    if hp_col:
        df[hp_col].dropna().astype(float).hist(bins=30, ax=axes[1], color='#C44E52', edgecolor='white')
        axes[1].set_title('Distribution de la puissance (HP)', fontweight='bold')
        axes[1].set_xlabel('Puissance (HP)')
        axes[1].set_ylabel('Nombre de véhicules')
    
    plt.tight_layout()
    plt.savefig('../results/exploration_distributions.png', dpi=150)
    plt.show()
    print('✅ Graphique sauvegardé')
else:
    print('Colonne seats non trouvée — vérifier les noms de colonnes du dataset')

In [ ]:
# Distribution du carburant
fuel_col = next((c for c in df.columns if 'fuel' in c.lower() or 'energy' in c.lower()), None)

if fuel_col:
    plt.figure(figsize=(8, 4))
    df[fuel_col].value_counts().plot(kind='bar', color='#55A868', edgecolor='white')
    plt.title('Distribution par type de carburant', fontweight='bold')
    plt.xlabel('Carburant')
    plt.ylabel('Nombre de véhicules')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 4. Analyse des corrélations

In [ ]:
# Scatter plot places vs puissance
if seat_col and hp_col:
    plot_df = df[[seat_col, hp_col]].dropna()
    plot_df.columns = ['seats', 'hp']
    plot_df['seats'] = pd.to_numeric(plot_df['seats'], errors='coerce')
    plot_df['hp'] = pd.to_numeric(plot_df['hp'], errors='coerce')
    plot_df = plot_df.dropna()
    
    plt.figure(figsize=(9, 5))
    plt.scatter(plot_df['seats'], plot_df['hp'], alpha=0.5, color='#4C72B0', edgecolors='white', linewidth=0.5)
    plt.xlabel('Nombre de places')
    plt.ylabel('Puissance (HP)')
    plt.title('Relation Places vs Puissance — indice de classification', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/scatter_seats_hp.png', dpi=150)
    plt.show()
    print('💡 Hypothèse : voitures à 2 places et haute puissance = sportives')
    print('   voitures à 5-7 places et puissance modérée = familiales')

## ✅ Conclusions de l'exploration

- Le dataset contient des informations structurées sur les voitures (places, HP, vitesse, carburant)
- On observe une tendance : **peu de places + haute puissance → sportive** / **beaucoup de places + puissance modérée → familiale**
- Ces attributs vont être transformés en texte pour alimenter le modèle zero-shot

➡️ **Passer au notebook 02_preprocessing.ipynb**